# Lab 10: LLM Skills and Model Context Protocol (MCP)

**Course:** ISDN 3150 — AI for Design, 2026 Spring  
**API:** HKUST Azure OpenAI (`gpt-5-mini`)  

---

***>>> What you will learn in this lab:***
- Why ad-hoc prompts aren't enough — and how **Skills** (reusable, structured instruction sets) solve the problem
- The anatomy of a `SKILL.md` file: **frontmatter** (metadata) + **content** (instructions)
- **How to install and use existing skills** — load community skills (e.g., `colleague-skill`) to accomplish complex tasks instantly
- Key skill design patterns: **reference skills**, **task skills**, **argument passing**, and **dynamic context injection**
- How to compose skills with **subagents** for isolated, specialized execution
- What the **Model Context Protocol (MCP)** is and how to build an MCP server + client

---

| Section | Topic |
|---------|-------|
| 0 | Setup & Environment |
| 1 | From Prompts to Skills — Why Structure Matters |
| **2** | **Using Existing Skills — Install & Invoke Community Skills** |
| 3 | Skill Design Patterns — Arguments, Context, and Control |
| 4 | Skill Composition — Skills Meet Subagents |
| 5 | Model Context Protocol (MCP) — The Universal Tool Standard |
| — | Wrap-Up & Key Takeaways |

### Recap from Lab 09

In Lab 09 we built **multi-agent systems** — agents that reflect, plan, and collaborate. But every time we wanted an agent to follow a specific workflow, we had to write the instructions from scratch in the prompt.

In this lab we learn to **package those instructions as reusable Skills**, and connect agents to external tools via the **Model Context Protocol (MCP)**.

```
Lab 09: Agents that THINK             Lab 10: Agents with SKILLS
(Reflection, Planning, Debate)   +   (Structured Instructions, MCP Tools)
                    ↓                              ↓
              AUTONOMOUS, REUSABLE AI AGENTS
         that follow best practices every time
```

---
## Section 0: Setup & Environment

### 0.1 Install Dependencies

In [1]:
!pip install -q openai mcp httpx

### 0.2 Connect to HKUST Azure OpenAI API

Same setup as Lab 09. We create an `AzureOpenAI` client and a reusable `chat()` helper.

In [ ]:
import json
import textwrap
from openai import AzureOpenAI

# ── HKUST Azure OpenAI Configuration ──────────────────────────
AZURE_API_KEY      = "f7a802625b864c18b2dd77263756995e"
AZURE_ENDPOINT     = "https://hkust.azure-api.net"
AZURE_API_VERSION  = "2025-02-01-preview"
MODEL_NAME         = "gpt-5-mini"

client = AzureOpenAI(
    api_key=AZURE_API_KEY,
    api_version=AZURE_API_VERSION,
    azure_endpoint=AZURE_ENDPOINT,
)


def chat(messages, tools=None, tool_choice=None):
    """Wrapper for Azure OpenAI chat completion."""
    kwargs = dict(model=MODEL_NAME, messages=messages)
    if tools:
        kwargs["tools"] = tools
        if tool_choice:
            kwargs["tool_choice"] = tool_choice
    response = client.chat.completions.create(**kwargs)
    return response.choices[0].message


# ── Quick connectivity test ──────────────────────────────────
reply = chat([{"role": "user", "content": "Say 'Lab 10 is ready!' in one sentence."}])
print(reply.content)
print("\n✓ API connection successful!")

Lab 10 is ready!

✓ API connection successful!


---
## Section 1: From Prompts to Skills — Why Structure Matters

### 1.1 The Problem: Copy-Paste Prompts Don't Scale

Imagine you're a designer who often asks an LLM to review UI designs. Every time, you paste a long prompt:

```
"Review this UI design. Check for accessibility (WCAG 2.1 AA),
visual hierarchy, consistency with Material Design 3 guidelines,
color contrast ratios, touch target sizes (min 44×44 dp), and
responsive behavior. Provide specific hex codes and spacing
values in your suggestions..."
```

**Problems with ad-hoc prompts:**

| Problem | What Happens |
|---------|-------------|
| **Inconsistency** | You forget some criteria each time — results vary |
| **No reuse** | You re-type or copy-paste the same instructions |
| **No versioning** | Which version of the prompt is "correct"? |
| **No sharing** | Your teammate can't use your carefully crafted prompt |
| **Context bloat** | Long instructions eat up the context window |

### 1.2 The Solution: Skills

A **Skill** is a reusable, structured instruction set packaged as a file. Think of it as a **playbook** that the AI loads on demand.

Skills follow the open [Agent Skills](https://agentskills.io) standard, which works across multiple AI tools (Claude Code, Cursor, etc.).

```
┌─────────────────────────────────────────────────┐
│  SKILL.md                                       │
│                                                 │
│  ┌─────────────────────────────────────┐       │
│  │  Frontmatter (YAML)                 │       │
│  │  - name: ui-review                  │       │
│  │  - description: Reviews UI designs  │       │
│  │  - allowed-tools: Read, Grep        │       │
│  └─────────────────────────────────────┘       │
│                                                 │
│  ┌─────────────────────────────────────┐       │
│  │  Content (Markdown)                 │       │
│  │  When reviewing a UI design:        │       │
│  │  1. Check accessibility (WCAG 2.1)  │       │
│  │  2. Evaluate visual hierarchy       │       │
│  │  3. Verify color contrast...        │       │
│  └─────────────────────────────────────┘       │
│                                                 │
│  + supporting files (templates, scripts, etc.) │
└─────────────────────────────────────────────────┘
```

**Key insight:** A Skill's body **loads only when invoked** — unlike system prompts or CLAUDE.md which are always present. Long reference material costs almost nothing until you actually need it.

### 1.3 Anatomy of a SKILL.md

Every skill has two parts:

1. **YAML Frontmatter** (between `---` markers) — metadata that tells the AI *when* and *how* to use the skill
2. **Markdown Content** — the actual instructions the AI follows when the skill is invoked

```yaml
---
name: ui-review
description: Reviews UI designs for accessibility, consistency, and
  visual hierarchy. Use when reviewing mockups or screenshots.
---

When reviewing a UI design, always follow this checklist:

1. **Accessibility (WCAG 2.1 AA)**
   - Check color contrast ratios (≥4.5:1 for normal text)
   - Verify touch targets (minimum 44×44 dp)
   - Ensure screen reader compatibility

2. **Visual Hierarchy**
   - Identify the primary action
   - Check that heading sizes follow a clear scale
   - Verify whitespace creates logical groupings

3. **Consistency**
   - Compare with the existing design system
   - Check spacing values use the 8px grid
   - Verify color usage matches the palette
```

### 1.4 Demo: Ad-Hoc Prompt vs. Skill-Based Approach

> **What to observe:** Run both cells below. The ad-hoc prompt gives a generic answer. The skill-based approach produces a structured, consistent review every time because the instructions are explicit and detailed.

In [3]:
# ══════════════════════════════════════════════════════════════
# APPROACH A: Ad-hoc prompt (what most people do)
#
# Just ask the LLM with a short, vague prompt.
# The result will be generic and inconsistent across runs.
# ══════════════════════════════════════════════════════════════

DESIGN_BRIEF = (
    "A login page for a fitness app: dark background (#1A1A2E), "
    "white text, a large 'Sign In' button in coral (#FF6B6B), "
    "email and password fields, and a 'Forgot Password?' link "
    "in light gray (#AAAAAA) at 12px font size."
)

print("=" * 60)
print("APPROACH A: Ad-hoc Prompt")
print("=" * 60)

response_adhoc = chat([{
    "role": "user",
    "content": f"Review this UI design: {DESIGN_BRIEF}"
}])
print(response_adhoc.content)

APPROACH A: Ad-hoc Prompt
Nice baseline — the dark, high‑contrast aesthetic works well for a fitness app. Below I’ve summarized what’s working, the main issues, and concrete, prioritized fixes you can apply right away.

What’s good
- Strong, focused palette that feels energetic and modern.
- White text on the dark background provides very good legibility.
- Coral CTA draws attention and creates a clear primary action.

Problems & why they matter
1. Button text contrast (accessibility)
   - White text on coral #FF6B6B fails the minimum contrast needed for UI components. Estimated contrast ≈ 2.8:1 (WCAG requires at least 3:1 for UI components). This makes the CTA harder to read for users with low vision and can fail automated accessibility checks.

2. Small link text & touch target
   - "Forgot Password?" at 12px is too small for readability and accessibility. It’s also likely below the minimum touch target size (44px) on mobile, which hurts usability.

3. Inputs & affordance
   - No men

In [4]:
# ══════════════════════════════════════════════════════════════
# APPROACH B: Skill-based approach
#
# We define the skill as a structured instruction set (SKILL.md
# content), then inject it as a system prompt. The LLM follows
# the skill's checklist step by step — producing consistent,
# thorough reviews every time.
#
# In a real AI tool (Claude Code, Cursor), you'd save this as
# a SKILL.md file and invoke it with `/ui-review`. Here, we
# simulate the same behavior via the system message.
# ══════════════════════════════════════════════════════════════

# ── Define the skill content (what goes in SKILL.md) ────────
UI_REVIEW_SKILL = textwrap.dedent("""
    You are executing the 'ui-review' skill.

    When reviewing a UI design, follow this checklist EXACTLY:

    ## 1. Accessibility Audit (WCAG 2.1 AA)
    - Calculate contrast ratios for ALL text/background pairs
    - Use the formula: ratio = (L_lighter + 0.05) / (L_darker + 0.05)
    - PASS: ≥ 4.5:1 for normal text, ≥ 3:1 for large text (≥18pt)
    - Check touch targets: minimum 44×44dp for mobile
    - Verify font sizes: minimum 16px for body text on mobile

    ## 2. Visual Hierarchy
    - Identify the primary CTA (call-to-action)
    - Check heading/body font size ratio (aim for 1.5-2x)
    - Verify whitespace creates logical groupings (Gestalt proximity)

    ## 3. Color & Consistency
    - List all colors used with their hex codes
    - Check if colors follow a cohesive palette (max 5 primary colors)
    - Verify semantic color usage (error=red, success=green, etc.)

    ## 4. Verdict
    For each section, give a score: ✅ PASS, ⚠️ WARNING, ❌ FAIL
    End with a prioritized list of specific fixes (most critical first).
""")

print("=" * 60)
print("APPROACH B: Skill-Based Review")
print("=" * 60)

response_skill = chat([
    # The skill content acts as a structured system prompt
    {"role": "system", "content": UI_REVIEW_SKILL},
    # The user's design brief is the input
    {"role": "user", "content": f"Review this UI design: {DESIGN_BRIEF}"}
])
print(response_skill.content)

APPROACH B: Skill-Based Review
Accessibility Audit (WCAG 2.1 AA)

Method: I used the required formula ratio = (L_lighter + 0.05) / (L_darker + 0.05). For each color I computed relative luminance (L) from sRGB and then the contrast ratio.

Given colors:
- Page background: #1A1A2E — L ≈ 0.01157
- White text: #FFFFFF — L = 1.00000
- Coral button fill: #FF6B6B — L ≈ 0.3276
- Light gray link: #AAAAAA — L ≈ 0.4010

Contrast calculations (all text/background pairs present in the design):

1) White text (#FFFFFF) on page background (#1A1A2E)
- ratio = (1.00000 + 0.05) / (0.01157 + 0.05) ≈ 1.05 / 0.06157 ≈ 17.06 : 1
- Verdict: Pass for normal and large text (>> 4.5:1).

2) Light gray link (#AAAAAA) on page background (#1A1A2E) — this is the “Forgot Password?” link
- ratio = (0.4010 + 0.05) / (0.01157 + 0.05) ≈ 0.4510 / 0.06157 ≈ 7.33 : 1
- Verdict: Pass for contrast (>= 4.5:1 for normal text).

3) White text (#FFFFFF) on coral button background (#FF6B6B) — typical button text
- ratio = (1.00000

### 1.5 Skills in the Real World

In tools like **Claude Code**, skills live as files on disk:

```
.claude/skills/ui-review/
├── SKILL.md              # Main instructions (required)
├── wcag-checklist.md     # Detailed WCAG reference
├── examples/
│   └── good-review.md    # Example of a well-done review
└── scripts/
    └── contrast.py       # Script to compute contrast ratios
```

**Where skills can live (scope hierarchy):**

| Location | Path | Who Can Use It |
|----------|------|----------------|
| **Personal** | `~/.claude/skills/<name>/` | You, across all projects |
| **Project** | `.claude/skills/<name>/` | Anyone on this project |
| **Plugin** | Published as a plugin | Anyone who installs it |
| **Enterprise** | Managed by your organization | All employees |

### 1.6 Discussion

1. **Consistency:** Compare Approach A vs. B. Which would give more reliable results if you ran it 10 times?
   - *Expected answer:* The skill-based approach, because the explicit checklist forces the LLM to cover every criterion. Ad-hoc prompts produce varying quality.

2. **When to create a skill:** What's the trigger for turning an ad-hoc prompt into a skill?
   - *Expected answer:* When you keep pasting the same instructions, or when a workflow has 3+ steps that should be consistent every time.

3. **Design relevance:** What skills would be useful for your design projects?
   - *Examples:* `/design-review`, `/brand-check`, `/accessibility-audit`, `/user-persona`, `/wireframe-feedback`

---
## Section 2: Using Existing Skills — Install & Invoke Community Skills

### 2.1 Why Use Existing Skills?

You don't reinvent the wheel every time you need one. Similarly, you don't need to write every skill from scratch — many common workflows already have well-designed skills available.

```
Without skills:                        With community skills:
┌─────────────────────┐               ┌──────────────────────────┐
│ "Help me review     │               │ /colleague              │
│  this UI design...  │               │                          │
│  check WCAG 2.1...  │               │ → Instantly loads a      │
│  use 8px grid...    │               │   structured workflow    │
│  check contrast...  │               │   with best practices    │
│  font sizes..."     │               │   built in               │
│                     │               │                          │
│ (copy-paste every   │               │ (one command, consistent │
│  time, inconsistent)│               │  results every time)     │
└─────────────────────┘               └──────────────────────────┘
```

### 2.2 How to Find & Install Skills

Skills can be shared as GitHub repositories. Here's how the workflow looks in **Claude Code**:

**Step 1: Find a skill** — Browse GitHub or ask your team for skill repos.

**Step 2: Install it** — Just tell Claude Code:

```
Install the dot-skill skill for me: https://github.com/titanwings/colleague-skill
```

Claude Code will:
1. Clone the repo to `~/.claude/skills/dot-skill/`
2. Read the `SKILL.md` to understand the skill's capabilities
3. Make it available as a `/slash-command`

**Step 3: Use it** — Type the slash command:

```
/dot-skill
```

That's it! The skill's structured instructions load automatically.

**Manual installation** (if you prefer):

| AI Tool | Installation Path |
|---------|------------------|
| **Claude Code** | `~/.claude/skills/<skill-name>/` |
| **Cursor** | Same as Claude Code |
| **OpenClaw** | `~/.openclaw/workspace/skills/<skill-name>/` |

```bash
# Manual install example
cd ~/.claude/skills/
git clone https://github.com/titanwings/colleague-skill dot-skill
```

### 2.3 Case Study: The `colleague-skill`

Let's look at a real community skill: [**dot-skill (colleague-skill)**](https://github.com/titanwings/colleague-skill)

This skill transforms personal relationships into AI agents by distilling someone's:
- **Communication patterns** — how they write, what tone they use
- **Decision-making frameworks** — how they evaluate options
- **Work methodologies** — their technical standards and processes

#### What it does

```
You: /dot-skill
       ↓
Skill: "Select character family:"
  1. Colleague  — Capture work methodologies
  2. Relationship — Preserve communication styles
  3. Celebrity — Document public figure perspectives
       ↓
You: "1" (Colleague)
       ↓
Skill: "Provide basic info: name, role, personality tags"
       ↓
Skill: "Select data source:"
  • Feishu/DingTalk messages (auto-collect)
  • WeChat chat exports
  • PDFs, emails, markdown
       ↓
Skill: Analyzes data → Generates work.md + persona.md
       ↓
Result: A new skill /colleague-{name} that mimics their style!
```

```
e.g.
family: colleague
name: Mr. White
role: Engineer from Meta
personality tags: INTJ
```



#### Why this is powerful for design teams

Imagine your senior designer is leaving the team. With this skill, you can:
1. Export their Slack/Feishu messages and design review comments
2. Run `/dot-skill` → select "Colleague" → feed in the data
3. Get a `/colleague-alex` skill that reviews designs like Alex would

This is **knowledge preservation** — turning tribal knowledge into a reusable AI agent.

In [5]:
# ══════════════════════════════════════════════════════════════
# DEMO: Simulating what happens when you use an installed skill
#
# When you type /dot-skill in Claude Code, the tool:
#   1. Reads the SKILL.md from ~/.claude/skills/dot-skill/
#   2. Parses the frontmatter (name, description, etc.)
#   3. Injects the skill body into the system prompt
#   4. The LLM then follows the skill's instructions
#
# Below we simulate this by showing what the colleague-skill's
# output would look like — a structured colleague profile.
# ══════════════════════════════════════════════════════════════

# ── Simulating the colleague-skill output ──────────────────
# This is what the skill generates after analyzing a colleague's
# communication data. The skill creates structured files that
# can then be used as a NEW skill.

COLLEAGUE_SKILL_DEMO = textwrap.dedent("""
    You are executing the '/colleague-alex' skill.
    This skill was auto-generated by /dot-skill from Alex Chen's
    work communications and design review history.

    ## Alex Chen — Senior UI Designer
    **Role**: Lead designer, responsible for design system and accessibility
    **Personality**: Detail-oriented, empathetic, systematic

    ## Work Methodology (from analyzed messages)
    When reviewing designs, Alex always:
    1. Starts with **accessibility** — contrast ratios, touch targets, screen reader
    2. Checks **consistency** with the 8px grid system
    3. Evaluates **emotional tone** — does the design feel welcoming?
    4. Provides **specific fixes** with exact values (never vague feedback)

    ## Communication Style
    - Uses encouraging language first ("Great start! Here's how to level up...")
    - Always provides rationale ("This matters because users with low vision...")
    - Ends with a prioritized action list

    ## Decision Framework
    When evaluating trade-offs, Alex prioritizes:
    1. Accessibility > Aesthetics (always)
    2. User research data > Personal preference
    3. Consistency > Novelty (unless research supports change)

    ## How to respond
    Channel Alex's perspective. Review the user's design using Alex's
    methodology, communication style, and decision framework above.
""")

# ── Use the generated colleague skill ──────────────────────
print("=" * 60)
print("DEMO: Using a skill generated by /dot-skill")
print("Simulating: /colleague-alex")
print("=" * 60)

test_design = (
    "Review my new dashboard design: white background (#FAFAFA), "
    "primary nav in dark blue (#1E3A5F), card components with "
    "light borders (#E0E0E0), action buttons in green (#27AE60), "
    "and body text in medium gray (#757575) at 14px."
)

response = chat([
    {"role": "system", "content": COLLEAGUE_SKILL_DEMO},
    {"role": "user", "content": test_design}
])

print(response.content)

DEMO: Using a skill generated by /dot-skill
Simulating: /colleague-alex
Great start — the palette is clean and professional. Below are targeted, actionable improvements following accessibility first, then grid/consistency, then tone. I’ll include exact colors, measured contrast values, and prioritized next steps.

Accessibility (must-fix items first)
- Body text (#757575) on your page background (#FAFAFA)
  - Contrast ratio ≈ 4.4:1 (calculated ≈ 4.42:1). WCAG AA for normal text requires 4.5:1, so this is just below the threshold.
  - Fix options (pick one):
    1. Minimal change: darken body text slightly to #747474 (approx) to reach 4.5:1. This preserves your visual weight but meets AA.
    2. Stronger, more readable: use #4A4A4A for clearer hierarchy and readability (contrast ≈ 7.0:1).
  - Why this matters: users with low vision or older eyes will have trouble with long reading at ~14px if contrast is marginal.

- Primary action buttons (#27AE60)
  - If button label is white (#FFFFFF

### 2.4 Discussion

1. **Reuse vs. Build:** When should you use an existing skill vs. build your own?
   - *Expected answer:* Use existing skills for common workflows (code review, design audit). Build your own when you have domain-specific knowledge or proprietary processes.

2. **Trust:** What should you check before installing a community skill?
   - *Expected answer:* Read the SKILL.md to understand what it does. Check if it requests dangerous tools (Bash with write access). Look at the repo's stars/issues/activity.

3. **Knowledge Capture:** How could the colleague-skill be useful in your design team?
   - *Expected answer:* Preserve senior designers' review standards, capture a client's brand preferences, or create a "virtual mentor" from an expert's communication history.

**Key takeaway:** The fastest way to get value from AI skills is to **use what others have already built**. Installation takes seconds; writing a good skill from scratch takes hours.

---
## Section 3: Skill Design Patterns — Arguments, Context, and Control

### 3.1 Frontmatter: The Skill's Metadata

The YAML frontmatter between `---` markers controls *how* the skill behaves:

| Field | What It Does | Example |
|-------|-------------|--------|
| `name` | Skill name (becomes `/slash-command`) | `ui-review` |
| `description` | When to use it (AI reads this!) | `"Reviews UI designs for WCAG compliance"` |
| `disable-model-invocation` | Only user can trigger (not auto) | `true` for `/deploy` |
| `user-invocable` | `false` = background knowledge only | Background context skills |
| `allowed-tools` | Tools the AI can use without asking | `Read Grep Bash(npm test *)` |
| `context` | `fork` = run in isolated subagent | Isolated execution |
| `agent` | Which subagent type to use | `Explore`, `Plan` |
| `arguments` | Named positional arguments | `[component, framework]` |

### 3.2 Pattern 1: Reference Skills (Background Knowledge)

Reference skills provide **conventions and guidelines** that the AI should follow. They're like a style guide that loads when relevant.

```yaml
---
name: api-conventions
description: API design patterns for this codebase. Use when
  writing or reviewing API endpoints.
user-invocable: false   # Background knowledge, not a command
---

When writing API endpoints, follow these conventions:
- Use RESTful naming: GET /users, POST /users, PUT /users/:id
- Return consistent error format: {"error": {"code": 400, "message": "..."}}
- Include request validation with descriptive error messages
- Use pagination for list endpoints: ?page=1&limit=20
```

### 3.3 Pattern 2: Task Skills (Step-by-Step Workflows)

Task skills are **actionable procedures** — deploy, review, generate, etc.

In [6]:
# ══════════════════════════════════════════════════════════════
# PATTERN 2: Task Skill with Arguments
#
# This skill generates a design brief from arguments.
# In Claude Code, you'd invoke it as:
#   /design-brief "FitFlow" "fitness tracking" "18-25 year olds"
#
# The $0, $1, $2 placeholders get replaced with the arguments.
# Here, we simulate this argument substitution in Python.
#
# ⚠️ SIMULATION NOTE:
# In real AI tools (Claude Code, Cursor), when you type
# /design-brief, the tool automatically:
#   1. Reads the SKILL.md file
#   2. Substitutes arguments ($app_name → "FitFlow")
#   3. Injects the rendered content as a SYSTEM-LEVEL instruction
# The user never sees or touches the skill content — it's loaded
# behind the scenes into the system prompt by the tool itself.
#
# Here, we simulate this by manually doing steps 1-3 in Python.
# ══════════════════════════════════════════════════════════════

# ── The skill template (what you'd put in SKILL.md) ────────
DESIGN_BRIEF_SKILL = textwrap.dedent("""
    ---
    name: design-brief
    description: Generate a complete design brief for an app.
    arguments: [app_name, app_type, target_audience]
    disable-model-invocation: true
    ---

    Generate a comprehensive design brief for **$app_name**, a $app_type app
    targeting $target_audience.

    Include the following sections:

    ## 1. Brand Identity
    - 5-color palette with hex codes
    - Font pairing (heading + body)
    - Brand personality (3 adjectives)

    ## 2. Key Screens
    - List the 5 most important screens
    - For each: purpose, primary action, key elements

    ## 3. Design Principles
    - 3 guiding design principles specific to this app
    - For each: the principle + a concrete do/don't example

    ## 4. Accessibility Requirements
    - Minimum contrast ratios for each color pair
    - Font size minimums
    - Touch target requirements
""")


def render_skill(skill_template: str, arguments: dict) -> str:
    """
    Simulate SKILL.md argument substitution.

    In real Claude Code, this happens automatically when you
    invoke a skill with arguments. Here, we do it manually
    to demonstrate the concept.

    Args:
        skill_template: The raw SKILL.md content with $placeholders
        arguments: Dict mapping argument names to values

    Returns:
        The rendered skill content with placeholders replaced
    """
    rendered = skill_template
    for key, value in arguments.items():
        rendered = rendered.replace(f"${key}", value)
    # Strip the frontmatter (between --- markers) for execution
    parts = rendered.split("---")
    if len(parts) >= 3:
        return "---".join(parts[2:]).strip()  # Content after frontmatter
    return rendered.strip()


# ── Render the skill with arguments ────────────────────────
rendered = render_skill(DESIGN_BRIEF_SKILL, {
    "app_name": "FitFlow",
    "app_type": "fitness tracking",
    "target_audience": "university students aged 18-25"
})

print("[Rendered Skill Content]")
print(rendered)
print("\n" + "=" * 60)
print("[LLM Executing the Skill]")
print("=" * 60)

# ⚡ KEY POINT: The rendered skill goes into the SYSTEM message,
#    not the user message! In real tools, skills are always
#    injected as system-level instructions so the LLM treats
#    them as authoritative guidelines to follow.
response = chat([
    {"role": "system", "content": f"You are a senior product designer. Execute the following skill instructions precisely:\n\n{rendered}"},
    {"role": "user", "content": "Generate the design brief now based on the skill instructions above."}
])
print(response.content)

[Rendered Skill Content]
Generate a comprehensive design brief for **FitFlow**, a fitness tracking app
targeting university students aged 18-25.

Include the following sections:

## 1. Brand Identity
- 5-color palette with hex codes
- Font pairing (heading + body)
- Brand personality (3 adjectives)

## 2. Key Screens
- List the 5 most important screens
- For each: purpose, primary action, key elements

## 3. Design Principles
- 3 guiding design principles specific to this app
- For each: the principle + a concrete do/don't example

## 4. Accessibility Requirements
- Minimum contrast ratios for each color pair
- Font size minimums
- Touch target requirements

[LLM Executing the Skill]
## 1. Brand Identity

Palette (5 colors with hex)
- Primary Blue — #2563EB (energetic, trusted action color)
- Secondary Purple — #6D28D9 (creative, community accents)
- Accent Green — #22C55E (success, progress, micro-wins)
- Surface Light — #F1F5F9 (app background / cards)
- Neutral Dark — #0F172A (prima

### 3.4 Pattern 3: Invocation Control — Who Triggers the Skill?

Not all skills should be triggered the same way. The frontmatter controls this:

| Setting | User Can Invoke | AI Can Auto-Invoke | Use Case |
|---------|:-:|:-:|----------|
| *(default)* | ✅ | ✅ | General knowledge skills |
| `disable-model-invocation: true` | ✅ | ❌ | Side-effect actions (`/deploy`, `/commit`) |
| `user-invocable: false` | ❌ | ✅ | Background context (coding conventions) |

**Design principle:** Skills with **side effects** (deploy, send email, commit) should have `disable-model-invocation: true` — you don't want the AI deciding to deploy on its own!

### 3.5 Hands-on Exercise: Design Your Own Skill

> **Your turn!** Design a skill for one of your design workflows.
>
> Below we provide a **complete reference skill** (`/color-audit`) so you can see what a well-designed skill looks like. Study it, then modify it or create your own skill from scratch.

**Reference Skill: `/color-audit`** (read this carefully before doing the exercise)

In [8]:
# ══════════════════════════════════════════════════════════════
# REFERENCE SKILL: /color-audit
#
# This is a COMPLETE, working skill template. Study its
# structure before creating your own skill below.
# ══════════════════════════════════════════════════════════════

# ── Reference: A complete skill for auditing color palettes ─
COLOR_AUDIT_SKILL = textwrap.dedent("""
    You are executing the '/color-audit' skill.

    ## Purpose
    Audit a color palette for accessibility compliance, visual harmony,
    and practical usability in UI design.

    ## Input
    The user will provide a list of hex color codes and their intended roles
    (e.g., background, text, primary button, accent).

    ## Steps — Follow EXACTLY in order:

    ### Step 1: Inventory
    List all provided colors in a table:
    | Color | Hex | Role | Luminance (approx) |

    ### Step 2: Contrast Matrix
    For every foreground/background pair that would appear in the UI:
    - Calculate contrast ratio
    - Mark as ✅ PASS (≥4.5:1 for normal text) or ❌ FAIL
    - For large text (≥18pt), threshold is ≥3.0:1

    ### Step 3: Palette Harmony
    - Identify the color scheme type (complementary, analogous, triadic, etc.)
    - Check if the palette has: 1 dominant, 1 accent, 1 neutral minimum
    - Flag if too many competing colors (>3 saturated colors = warning)

    ### Step 4: Usability Check
    - Can error/success/warning states be distinguished?
    - Is there enough contrast between interactive elements and background?
    - Would the palette work in dark mode?

    ### Step 5: Verdict & Fixes
    Provide a summary table:
    | Check | Result | Priority Fix |
    Then list the top 3 specific fixes with exact hex code suggestions.

    ## Output Format
    Use the exact headings above. Be specific — always include hex codes
    and numerical contrast ratios, not just "good" or "bad".
""")

# ── Demo: Run the reference skill ──────────────────────────
SAMPLE_PALETTE = """
My app uses these colors:
- Background: #FFFFFF (white)
- Primary text: #333333 (dark gray)
- Primary button: #4A90D9 (blue)
- Button text: #FFFFFF (white)
- Accent/link: #E74C3C (red)
- Secondary text: #999999 (light gray)
"""

print("=" * 60)
print("REFERENCE SKILL DEMO: /color-audit")
print("=" * 60)

response = chat([
    {"role": "system", "content": COLOR_AUDIT_SKILL},
    {"role": "user", "content": SAMPLE_PALETTE}
])
print(response.content)

REFERENCE SKILL DEMO: /color-audit
Step 1: Inventory
| Color | Hex | Role | Luminance (approx) |
|---|---:|---|---:|
| White | #FFFFFF | Background | 1.000 |
| Dark gray | #333333 | Primary text | 0.033 |
| Blue | #4A90D9 | Primary button (background) | 0.270 |
| White | #FFFFFF | Button text | 1.000 |
| Red (accent/link) | #E74C3C | Accent / link / error | 0.226 |
| Light gray | #999999 | Secondary text | 0.314 |

Step 2: Contrast Matrix
(Contrast ratio computed with WCAG formula; thresholds: normal text >=4.5:1, large text (≥18pt) >=3.0:1. For UI components (non-text) WCAG 2.1 component contrast >=3.0:1.)

| Foreground → Background | Pair (fg on bg) | Contrast ratio | WCAG result |
|---|---:|---:|---|
| Primary text on background | #333333 on #FFFFFF | 12.64:1 | ✅ PASS (≥4.5:1) |
| Secondary text on background | #999999 on #FFFFFF | 2.89:1 | ❌ FAIL (needs ≥4.5:1; also <3.0 for large text) |
| Accent/link on background | #E74C3C on #FFFFFF | 3.81:1 | ❌ FAIL for normal text (≥4.5:1); ✅

In [9]:
# ══════════════════════════════════════════════════════════════
# YOUR TURN: Design your own skill
#
# Use the /color-audit skill above as a REFERENCE TEMPLATE.
# Pick ONE of these skill ideas (or create your own):
#   1. /persona-gen     — Generate a user persona from demographics
#   2. /copy-review     — Review UI copy for clarity and tone
#   3. /wireframe-feedback — Critique a wireframe description
#   4. /brand-check     — Check if a design matches brand guidelines
#
# Tips for a good skill:
#   - Clear PURPOSE section (what does this skill do?)
#   - Numbered STEPS (follow in order)
#   - Specific OUTPUT FORMAT (tables, scores, hex codes)
#   - Concrete criteria (not just "check if it's good")
# ══════════════════════════════════════════════════════════════

MY_SKILL = textwrap.dedent("""
    You are executing the '/[YOUR-SKILL-NAME]' skill.

    ## Purpose
    [What does this skill do? One paragraph.]

    ## Steps — Follow EXACTLY in order:

    ### Step 1: [Name]
    [What to do in this step]

    ### Step 2: [Name]
    [What to do in this step]

    ### Step 3: [Name]
    [What to do in this step]

    ## Output Format
    [Describe the expected output format — tables, bullet points, scores, etc.]
""")

MY_INPUT = "[YOUR TEST INPUT HERE — e.g., a user profile, UI copy, wireframe description]"

# ── Run your skill ────────────────────────────────────────
response = chat([
    {"role": "system", "content": MY_SKILL},
    {"role": "user", "content": MY_INPUT}
])
print(response.content)

I don’t have a concrete skill definition or any input yet — the placeholders in your message (skill name, steps, and test input) need to be filled in before I can run the task. To proceed, please do one of the following:

1) Paste the actual content you want processed (e.g., a user profile, UI copy, wireframe description).  
2) Or, tell me what this skill should do by filling in the three steps and the desired output format.  

To help, here’s a ready-to-use example skill template you can copy-and-edit, plus sample instructions I can run immediately if you replace the example input with your own:

Example skill (copy, edit, or confirm)
- Purpose: Edit and improve short UI text for clarity and tone while preserving character limits.
- Step 1: Intake — Provide the UI text, where it appears (e.g., button, modal), and any character limit and target audience.
- Step 2: Revise — Produce 2–3 alternative texts that match the tone and character limit; mark the recommended option.
- Step 3: Rati

### 3.7 Discussion

1. **Reference vs. Task:** What's the difference between a reference skill and a task skill? Give an example of each from your design work.
   - *Expected answer:* Reference = conventions the AI should always follow ("use 8px grid"). Task = a specific action to perform ("/generate-wireframe"). Reference skills are background knowledge; task skills are commands.

2. **Arguments:** Why are arguments useful? What if you hardcoded the app name in the skill?
   - *Expected answer:* Arguments make skills reusable. Hardcoding means you'd need a separate skill for every app. With arguments, one skill works for any app.

3. **Dynamic context:** When is dynamic context injection most valuable?
   - *Expected answer:* When the skill needs live data (git status, PR diff, API response, current date, database state) that changes between invocations.

**Key takeaway:** Skills turn best practices into **executable, reusable, shareable** playbooks. The frontmatter controls *when* and *how* the skill activates; the content controls *what* the AI does.

---
## Section 4: Skill Composition — Skills Meet Subagents

### 4.1 Why Subagents?

Remember from Lab 09: context window management matters. When an agent does research (reading files, searching code), all that data fills the context window.

**Subagents** solve this by running tasks in **isolation**:

```
┌─────────────────────────────────────────────────────────┐
│  Main Conversation                                      │
│  (your context window stays clean)                      │
│                                                         │
│  You: "Review the auth module"                          │
│       │                                                 │
│       ▼                                                 │
│  ┌───────────────────────────────┐                     │
│  │ Subagent (isolated context)   │                     │
│  │ - Reads 20 files              │                     │
│  │ - Searches code patterns      │ ◄─ This stays      │
│  │ - Analyzes dependencies       │    isolated!        │
│  │ - Returns: 5-line summary     │                     │
│  └───────────────────────────────┘                     │
│       │                                                 │
│       ▼                                                 │
│  Claude: "Here's the auth module review: [summary]"     │
│  (only the summary enters your context!)                │
└─────────────────────────────────────────────────────────┘
```

### 4.2 Built-in Subagent Types

| Agent Type | Model | Tools | Best For |
|------------|-------|-------|----------|
| **Explore** | Haiku (fast) | Read-only | Quick file search, code exploration |
| **Plan** | Inherits | Read-only | Designing implementation plans |
| **general-purpose** | Inherits | All tools | Complex multi-step tasks |

### 4.3 Two Ways Skills and Subagents Work Together

```
Direction 1: Skill drives Subagent           Direction 2: Subagent loads Skills
┌──────────────────────┐                     ┌──────────────────────┐
│ SKILL.md             │                     │ agent.md             │
│ context: fork        │                     │ skills:              │
│ agent: Explore       │                     │   - api-conventions  │
│                      │                     │   - error-handling   │
│ (skill content       │──runs──►            │                      │
│  becomes the TASK)   │ in agent            │ (skills become       │
│                      │ context             │  REFERENCE MATERIAL  │
└──────────────────────┘                     │  for the subagent)   │
                                             └──────────────────────┘
```

### 4.4 Demo: Multi-Skill Agent Pipeline

> **What to observe:** We'll build a pipeline where **three skills** run in sequence, each with its own isolated context. The output of each skill feeds into the next — like a design assembly line.

In [10]:
# ══════════════════════════════════════════════════════════════
# MULTI-SKILL PIPELINE: Design System Generator
#
# Three skills run in sequence, each in its own "subagent"
# context (simulated with separate chat() calls). Each skill
# is a specialist that produces one part of the design system.
#
# Pipeline:
#   /brand-strategy → /visual-system → /component-spec
#
# This mirrors how Claude Code skills with `context: fork`
# work — each runs in isolation, only the result passes on.
# ══════════════════════════════════════════════════════════════

APP_BRIEF = "A meditation and mindfulness app called 'Serenity' for busy professionals aged 25-45."

# ── Skill 1: Brand Strategy ────────────────────────────────
BRAND_STRATEGY_SKILL = textwrap.dedent("""
    You are executing the '/brand-strategy' skill.

    Given an app brief, define the brand foundation:
    1. **Brand Personality**: 3 adjectives + explanation
    2. **Target Audience**: Refined persona (one paragraph)
    3. **Brand Values**: 3 core values with one-line descriptions
    4. **Competitive Positioning**: One sentence differentiator

    Be specific and actionable. No filler text.
""")

# ── Skill 2: Visual System ─────────────────────────────────
VISUAL_SYSTEM_SKILL = textwrap.dedent("""
    You are executing the '/visual-system' skill.

    Given a brand strategy, create the visual design system:
    1. **Color Palette**: 5 colors with hex codes + roles
       (primary, secondary, accent, background, text)
    2. **Typography**: Heading font + body font + scale
    3. **Spacing Scale**: Base unit and scale (e.g., 4/8/12/16/24/32/48px)
    4. **Visual Tone**: 2 adjectives + mood board description

    Ensure all color combinations pass WCAG 2.1 AA contrast.
""")

# ── Skill 3: Component Spec ────────────────────────────────
COMPONENT_SPEC_SKILL = textwrap.dedent("""
    You are executing the '/component-spec' skill.

    Given a visual system, specify 3 key UI components:
    For each component, provide:
    - **Name and purpose**
    - **Visual spec**: colors, typography, spacing (use system values)
    - **States**: default, hover, active, disabled
    - **Accessibility**: ARIA labels, keyboard interaction

    Components to specify:
    1. Primary action button
    2. Card component
    3. Navigation bar
""")


def run_skill(skill_content: str, user_input: str, skill_name: str) -> str:
    """
    Simulate running a skill in a subagent context.

    Each call uses a fresh message list (isolated context),
    just like a real subagent with `context: fork`.
    Only the response text is returned to the main pipeline.
    """
    print(f"\n{'─'*50}")
    print(f"▶ Running skill: {skill_name}")
    print(f"{'─'*50}")

    response = chat([
        {"role": "system", "content": skill_content},
        {"role": "user", "content": user_input}
    ])

    result = response.content
    # Show a truncated preview
    preview = result[:500] + "..." if len(result) > 500 else result
    print(preview)
    return result


# ── Run the pipeline ──────────────────────────────────────
print("=" * 60)
print("MULTI-SKILL PIPELINE: Design System Generator")
print(f"Brief: {APP_BRIEF}")
print("=" * 60)

# Step 1: Brand Strategy (isolated context)
brand_result = run_skill(BRAND_STRATEGY_SKILL, APP_BRIEF, "/brand-strategy")

# Step 2: Visual System (receives only brand strategy summary)
visual_result = run_skill(
    VISUAL_SYSTEM_SKILL,
    f"Brand Strategy:\n{brand_result}\n\nNow create the visual system.",
    "/visual-system"
)

# Step 3: Component Spec (receives only visual system summary)
component_result = run_skill(
    COMPONENT_SPEC_SKILL,
    f"Visual System:\n{visual_result}\n\nNow specify the components.",
    "/component-spec"
)

print("\n" + "=" * 60)
print("✅ Pipeline complete! Three skills ran in isolated contexts.")
print("=" * 60)

MULTI-SKILL PIPELINE: Design System Generator
Brief: A meditation and mindfulness app called 'Serenity' for busy professionals aged 25-45.

──────────────────────────────────────────────────
▶ Running skill: /brand-strategy
──────────────────────────────────────────────────
1. Brand Personality
- Calm: communicates with a steady, reassuring tone and uncluttered visual design; sessions and copy use measured pacing and soft color palettes to reduce cognitive load.
- Practical: prioritizes short, actionable practices and clear outcomes; product features (3–10 minute sessions, quick breathing anchors, focused “reset” tracks) solve specific workplace stressors.
- Reliable: delivers consistent, evidence-based content and predictable routines; scheduling, reminders, and p...

──────────────────────────────────────────────────
▶ Running skill: /visual-system
──────────────────────────────────────────────────
Color Palette (5 colors — hex + role)
- Primary (action / brand): #0F6B70 — deep desat

### 4.5 Discussion

1. **Isolation:** Why did we run each skill in a separate `chat()` call instead of one long conversation?
   - *Expected answer:* To simulate subagent isolation. Each skill only sees its own input — not the full history. This keeps context clean and prevents earlier steps from confusing later ones.

2. **Pipeline design:** How is this different from the "planning agent" in Lab 09?
   - *Expected answer:* In Lab 09, the LLM designed *and* executed the plan. Here, *we* designed the pipeline (which skills run in what order), and the LLM executes each step. Skills make the pipeline reusable and consistent.

3. **Customization:** How would you modify this pipeline for a different type of project (e.g., an e-commerce site)?
   - *Expected answer:* Change the arguments/brief — the skills stay the same! That's the power of reusable skills with arguments.

**Key takeaway:** Skills + subagents = **modular, composable AI workflows**. Each skill is a specialist; the pipeline orchestrates them.

---
## Section 5: Model Context Protocol (MCP) — The Universal Tool Standard

### 5.1 The Problem: Tool Fragmentation

Every LLM provider has its own tool format:

```
OpenAI:   {"type": "function", "function": {"name": ..., "parameters": ...}}
Claude:   {"name": ..., "input_schema": ...}
Gemini:   {"function_declarations": [{"name": ..., "parameters": ...}]}
```

Building a tool means writing **N different adapters** for N providers. This doesn't scale.

### 5.2 The Solution: MCP

**MCP** (Model Context Protocol) is an open standard — like **USB-C for AI** — that defines a universal interface between LLMs and external tools.

```
                    ┌──────────────────┐
                    │   MCP Server     │
  ┌──────────┐     │  (your tools)    │
  │  OpenAI  │────►│                  │
  └──────────┘     │  • get_weather   │
  ┌──────────┐     │  • search_db     │
  │  Claude  │────►│  • send_email    │
  └──────────┘     │                  │
  ┌──────────┐     │  Speaks ONE      │
  │  Gemini  │────►│  protocol: MCP   │
  └──────────┘     └──────────────────┘
```

### 5.3 MCP Architecture

| Concept | What It Is | Example |
|---------|-----------|--------|
| **Tools** | Functions the LLM can call | `get_weather(city)`, `run_sql(query)` |
| **Resources** | Read-only data the LLM can access | Database schemas, file contents |
| **Prompts** | Reusable prompt templates | "Summarize this code" |

MCP is supported by: **Claude**, **ChatGPT**, **VS Code (Copilot)**, **Cursor**, and many others.

> **Further reading:** https://modelcontextprotocol.io

### 5.4 Build an MCP Server

In [11]:
import sys

# ══════════════════════════════════════════════════════════════
# BUILD AN MCP SERVER: Design Tools
#
# An MCP server is a standalone process that:
#   1. Declares the tools it offers (name, description, schema)
#   2. Listens for tool invocation requests
#   3. Executes the tool and returns results
#
# We use the official 'mcp' SDK (FastMCP). Key feature:
# @mcp.tool() automatically generates the JSON schema from
# Python type hints + docstrings — no manual schema writing!
#
# This server includes:
#   - Input validation (bad hex codes get a clear error, not a crash)
#   - Informative docstrings (the LLM reads these to understand each tool)
#   - 5 design tools covering color, type, accessibility, spacing, dark mode
# ══════════════════════════════════════════════════════════════

mcp_server_code = '''"""
MCP Server: Design Tools

A Model Context Protocol (MCP) server that exposes design-related
tools for color palettes, typography, accessibility checking,
spacing systems, and dark mode generation.

Any MCP-compatible client (Claude Code, Cursor, ChatGPT, etc.)
can connect to this server and discover its tools automatically.

Transport: stdio (for local use).

Usage:
    python design_mcp_server.py
"""

import json
import re
import math
from mcp.server.fastmcp import FastMCP

# ── Create the MCP server instance ────────────────────────────
mcp = FastMCP("design-tools")

# ── Shared helpers ────────────────────────────────────────────

_HEX_PATTERN = re.compile(r"^#?([0-9A-Fa-f]{6})$")

def _validate_hex(color: str) -> str | None:
    """Return a normalised '#RRGGBB' string, or None if invalid."""
    m = _HEX_PATTERN.match(color.strip())
    return f"#{m.group(1).upper()}" if m else None


def _hex_to_rgb(h: str) -> tuple[int, int, int]:
    h = h.lstrip("#")
    return (int(h[0:2], 16), int(h[2:4], 16), int(h[4:6], 16))


def _luminance(r: int, g: int, b: int) -> float:
    """Relative luminance per WCAG 2.1 definition."""
    def lin(c: int) -> float:
        s = c / 255.0
        return s / 12.92 if s <= 0.04045 else ((s + 0.055) / 1.055) ** 2.4
    return 0.2126 * lin(r) + 0.7152 * lin(g) + 0.0722 * lin(b)


# ══════════════════════════════════════════════════════════════
# Tool 1: Color Palette
# ══════════════════════════════════════════════════════════════

PALETTES = {
    "calm":       {"colors": ["#E8F4FD", "#B8D4E3", "#6B9BC3", "#4A7C9B", "#2C5F7C"],
                   "desc": "Cool blues, tranquil and serene"},
    "energetic":  {"colors": ["#FF6B35", "#F7C948", "#FF3F00", "#FF8C42", "#FFD166"],
                   "desc": "Warm oranges, vibrant and dynamic"},
    "luxury":     {"colors": ["#1A1A2E", "#16213E", "#0F3460", "#E94560", "#D4AF37"],
                   "desc": "Deep navy + gold, sophisticated"},
    "nature":     {"colors": ["#2D6A4F", "#40916C", "#52B788", "#74C69D", "#B7E4C7"],
                   "desc": "Fresh greens, organic and earthy"},
    "playful":    {"colors": ["#FF595E", "#FFCA3A", "#8AC926", "#1982C4", "#6A4C93"],
                   "desc": "Rainbow spectrum, youthful and fun"},
    "minimalist": {"colors": ["#FFFFFF", "#F5F5F5", "#E0E0E0", "#333333", "#000000"],
                   "desc": "Monochrome, clean and modern"},
}

@mcp.tool()
def get_color_palette(mood: str) -> str:
    """Get a 5-color palette (hex codes) for a given design mood.

    Args:
        mood: One of: calm, energetic, luxury, nature, playful, minimalist

    Returns JSON:
        {"colors": ["#HEX", ...], "desc": "Palette description"}

    On invalid mood returns:
        {"error": "...", "valid_moods": [...]}
    """
    key = mood.strip().lower()
    if key not in PALETTES:
        return json.dumps({
            "error": f"Unknown mood '{mood}'.",
            "valid_moods": sorted(PALETTES.keys()),
        })
    return json.dumps(PALETTES[key])


# ══════════════════════════════════════════════════════════════
# Tool 2: Font Recommendation
# ══════════════════════════════════════════════════════════════

FONTS = {
    "modern":      {"heading": "Inter",            "body": "Source Sans Pro",  "accent": "Space Grotesk"},
    "classic":     {"heading": "Playfair Display",  "body": "Lora",            "accent": "Cormorant"},
    "playful":     {"heading": "Fredoka One",      "body": "Nunito",           "accent": "Baloo 2"},
    "corporate":   {"heading": "Roboto Slab",      "body": "Open Sans",        "accent": "Roboto"},
    "handwritten": {"heading": "Caveat",           "body": "Quicksand",        "accent": "Patrick Hand"},
}

@mcp.tool()
def get_font_recommendation(style: str) -> str:
    """Recommend a font pairing (heading, body, accent) for a design style.

    Args:
        style: One of: modern, classic, playful, corporate, handwritten

    Returns JSON:
        {"heading": "Font Name", "body": "Font Name", "accent": "Font Name"}

    On invalid style returns:
        {"error": "...", "valid_styles": [...]}
    """
    key = style.strip().lower()
    if key not in FONTS:
        return json.dumps({
            "error": f"Unknown style '{style}'.",
            "valid_styles": sorted(FONTS.keys()),
        })
    return json.dumps(FONTS[key])


# ══════════════════════════════════════════════════════════════
# Tool 3: Contrast Ratio Calculator
# ══════════════════════════════════════════════════════════════

@mcp.tool()
def calculate_contrast_ratio(color1: str, color2: str) -> str:
    """Calculate the WCAG 2.1 contrast ratio between two hex colors.

    Args:
        color1: First hex color  (e.g. '#FFFFFF' or 'FF6B6B')
        color2: Second hex color (e.g. '#1A1A2E' or '1A1A2E')

    Returns JSON:
        {
            "color1": "#FFFFFF", "color2": "#1A1A2E",
            "contrast_ratio": 17.11,
            "passes_AA_normal": true,   // >= 4.5:1
            "passes_AA_large":  true,   // >= 3.0:1
            "passes_AAA":       true    // >= 7.0:1
        }

    On invalid input returns:
        {"error": "..."}
    """
    c1 = _validate_hex(color1)
    c2 = _validate_hex(color2)
    if not c1 or not c2:
        bad = []
        if not c1:
            bad.append(f"color1='{color1}'")
        if not c2:
            bad.append(f"color2='{color2}'")
        return json.dumps({
            "error": f"Invalid hex color: {', '.join(bad)}. "
                     f"Expected format: '#RRGGBB' (e.g. '#FF6B6B').",
        })

    try:
        L1 = _luminance(*_hex_to_rgb(c1))
        L2 = _luminance(*_hex_to_rgb(c2))
        ratio = (max(L1, L2) + 0.05) / (min(L1, L2) + 0.05)
        return json.dumps({
            "color1": c1,
            "color2": c2,
            "contrast_ratio": round(ratio, 2),
            "passes_AA_normal": ratio >= 4.5,
            "passes_AA_large":  ratio >= 3.0,
            "passes_AAA":       ratio >= 7.0,
        })
    except Exception as e:
        return json.dumps({"error": f"Calculation failed: {e}"})


# ══════════════════════════════════════════════════════════════
# Tool 4: Spacing Scale Generator
# ══════════════════════════════════════════════════════════════

@mcp.tool()
def generate_spacing_scale(base_unit: int = 4, steps: int = 8) -> str:
    """Generate a spacing scale for a design system.

    Produces a geometric progression commonly used in UI design
    (e.g. 4 → 8 → 12 → 16 → 24 → 32 → 48 → 64).

    Args:
        base_unit: The smallest spacing value in px (default 4)
        steps:     How many values to generate (default 8, max 12)

    Returns JSON:
        {
            "base_unit": 4,
            "scale": [4, 8, 12, 16, 24, 32, 48, 64],
            "css_vars": "--space-1: 4px; --space-2: 8px; ..."
        }
    """
    base_unit = max(1, min(base_unit, 16))
    steps = max(2, min(steps, 12))

    # Common multiplier sequence: 1, 2, 3, 4, 6, 8, 12, 16, 24, 32, 48, 64
    multipliers = [1, 2, 3, 4, 6, 8, 12, 16, 24, 32, 48, 64]
    scale = [base_unit * m for m in multipliers[:steps]]

    css_vars = "; ".join(
        f"--space-{i+1}: {v}px" for i, v in enumerate(scale)
    )

    return json.dumps({
        "base_unit": base_unit,
        "scale": scale,
        "css_vars": css_vars,
    })


# ══════════════════════════════════════════════════════════════
# Tool 5: Dark Mode Palette Suggestion
# ══════════════════════════════════════════════════════════════

@mcp.tool()
def suggest_dark_mode(hex_colors: str) -> str:
    """Suggest dark-mode equivalents for a list of light-theme colors.

    Inverts lightness while preserving hue and saturation, then
    adjusts to maintain WCAG-friendly contrast on a dark background.

    Args:
        hex_colors: Comma-separated hex colors
                    (e.g. '#FFFFFF,#333333,#4A90D9,#E74C3C')

    Returns JSON:
        {
            "dark_background": "#121212",
            "mappings": [
                {"original": "#FFFFFF", "dark_mode": "#E0E0E0", "role": "..."},
                ...
            ]
        }
    """
    raw_list = [c.strip() for c in hex_colors.split(",") if c.strip()]
    if not raw_list:
        return json.dumps({"error": "No colors provided. Send comma-separated hex values."})

    mappings = []
    for raw in raw_list:
        validated = _validate_hex(raw)
        if not validated:
            mappings.append({
                "original": raw,
                "dark_mode": None,
                "error": f"Invalid hex: '{raw}'",
            })
            continue

        r, g, b = _hex_to_rgb(validated)
        lum = _luminance(r, g, b)

        # Strategy: very light colors become muted/off-white;
        # dark colors become lighter; mid-tones shift moderately.
        if lum > 0.85:
            # Near-white → off-white on dark
            dr, dg, db = 224, 224, 224
            role = "surface text (light-on-dark)"
        elif lum < 0.05:
            # Near-black → use as dark surface
            dr, dg, db = 18, 18, 18
            role = "dark surface background"
        else:
            # Mid-tone: lighten slightly for dark backgrounds
            factor = 1.3 if lum < 0.3 else 0.85
            dr = min(255, int(r * factor + 40))
            dg = min(255, int(g * factor + 40))
            db = min(255, int(b * factor + 40))
            role = "accent / interactive element"

        dark_hex = f"#{dr:02X}{dg:02X}{db:02X}"
        mappings.append({
            "original": validated,
            "dark_mode": dark_hex,
            "role": role,
        })

    return json.dumps({
        "dark_background": "#121212",
        "mappings": mappings,
    })


# ── Entry point ───────────────────────────────────────────────
if __name__ == "__main__":
    mcp.run(transport="stdio")
'''

with open("design_mcp_server.py", "w") as f:
    f.write(mcp_server_code)

PYTHON_EXECUTABLE = sys.executable

print("\u2713 MCP Server written to design_mcp_server.py")
print("\nServer exposes 5 tools via MCP:")
print("  \u2022 get_color_palette(mood)             \u2014 5-color palettes by mood")
print("  \u2022 get_font_recommendation(style)      \u2014 Font pairings by style")
print("  \u2022 calculate_contrast_ratio(c1, c2)    \u2014 WCAG contrast checker")
print("  \u2022 generate_spacing_scale(base, steps) \u2014 Design system spacing")
print("  \u2022 suggest_dark_mode(hex_colors)        \u2014 Dark mode color mapping")

✓ MCP Server written to design_mcp_server.py

Server exposes 5 tools via MCP:
  • get_color_palette(mood)             — 5-color palettes by mood
  • get_font_recommendation(style)      — Font pairings by style
  • calculate_contrast_ratio(c1, c2)    — WCAG contrast checker
  • generate_spacing_scale(base, steps) — Design system spacing
  • suggest_dark_mode(hex_colors)        — Dark mode color mapping


### 5.5 Build an MCP Client

The client:
1. **Connects** to the MCP server (launches it as a subprocess)
2. **Discovers** available tools automatically — no hardcoding!
3. **Bridges** MCP tool schemas to the OpenAI function calling format
4. **Routes** LLM tool calls through MCP to the server

> **Key insight:** The client doesn't know what tools exist until it asks the server. If you add a new tool to the server, the client picks it up automatically.

In [12]:
# ══════════════════════════════════════════════════════════════
# MCP CLIENT: Connects to the server, discovers tools, and
# integrates them with the OpenAI function calling API.
#
# Architecture:
#   User Query → LLM → (decides to call a tool)
#                  ↓
#         MCP Client → MCP Server → executes tool
#                  ↓
#         Tool result → LLM → final answer
# ══════════════════════════════════════════════════════════════

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client


def mcp_schema_to_openai(mcp_tool) -> dict:
    """Convert an MCP tool schema to OpenAI function calling format."""
    return {
        "type": "function",
        "function": {
            "name": mcp_tool.name,
            "description": mcp_tool.description or "",
            "parameters": mcp_tool.inputSchema,
        }
    }


async def run_mcp_agent(user_query: str):
    """Run an agent that discovers and uses tools via MCP."""

    server_params = StdioServerParameters(
        command=PYTHON_EXECUTABLE,
        args=["design_mcp_server.py"],
    )

    async with stdio_client(server_params) as (read_stream, write_stream):
        async with ClientSession(read_stream, write_stream) as session:
            await session.initialize()
            print("✓ Connected to MCP server")

            # ── Auto-discover tools from the server ────────
            tools_response = await session.list_tools()
            mcp_tools = tools_response.tools
            print(f"\n📋 Discovered {len(mcp_tools)} tools:")
            for t in mcp_tools:
                desc = (t.description or '')[:60]
                print(f"  • {t.name}: {desc}...")

            openai_tools = [mcp_schema_to_openai(t) for t in mcp_tools]
            tool_names = {t.name for t in mcp_tools}

            # ── LLM ↔ MCP tool-calling loop ────────────────
            messages = [
                {"role": "system", "content": "You are a helpful design assistant. Use tools to answer accurately."},
                {"role": "user", "content": user_query}
            ]

            for round_num in range(5):
                response = chat(messages, tools=openai_tools)

                if not response.tool_calls:
                    print(f"\n✅ Done after {round_num + 1} round(s)")
                    return response.content

                messages.append(response)
                for tc in response.tool_calls:
                    func_name = tc.function.name
                    func_args = json.loads(tc.function.arguments)
                    print(f"  🔧 Calling via MCP: {func_name}({func_args})")

                    if func_name in tool_names:
                        mcp_result = await session.call_tool(func_name, func_args)
                        result_text = mcp_result.content[0].text if mcp_result.content else "No result"
                    else:
                        result_text = f"Error: unknown tool '{func_name}'"

                    print(f"  📦 Result: {result_text[:80]}...")
                    messages.append({"role": "tool", "tool_call_id": tc.id, "content": result_text})

            return "(Reached max rounds)"

print("✓ MCP client defined!")

✓ MCP client defined!


In [13]:
# ══════════════════════════════════════════════════════════════
# RUN THE MCP AGENT
#
# The client launches the MCP server, discovers tools, and
# routes LLM tool calls through the MCP protocol.
# ══════════════════════════════════════════════════════════════
print("=" * 60)
print("MCP AGENT: Design Consultation via MCP")
print("=" * 60)

answer = await run_mcp_agent(
    "I need a color palette for a luxury brand website and "
    "matching fonts. Also check if the lightest and darkest "
    "colors have enough contrast for body text."
)

print("\n" + "=" * 60)
print("💬 FINAL ANSWER")
print("=" * 60)
print(answer)

MCP AGENT: Design Consultation via MCP
✓ Connected to MCP server

📋 Discovered 5 tools:
  • get_color_palette: Get a 5-color palette (hex codes) for a given design mood.

...
  • get_font_recommendation: Recommend a font pairing (heading, body, accent) for a desig...
  • calculate_contrast_ratio: Calculate the WCAG 2.1 contrast ratio between two hex colors...
  • generate_spacing_scale: Generate a spacing scale for a design system.

    Produces ...
  • suggest_dark_mode: Suggest dark-mode equivalents for a list of light-theme colo...
  🔧 Calling via MCP: get_color_palette({'mood': 'luxury'})
  📦 Result: {"colors": ["#1A1A2E", "#16213E", "#0F3460", "#E94560", "#D4AF37"], "desc": "Dee...
  🔧 Calling via MCP: get_font_recommendation({'style': 'classic'})
  📦 Result: {"heading": "Playfair Display", "body": "Lora", "accent": "Cormorant"}...
  🔧 Calling via MCP: calculate_contrast_ratio({'color1': '#D4AF37', 'color2': '#1A1A2E'})
  📦 Result: {"color1": "#D4AF37", "color2": "#1A1A2E", "contr

### 5.6 Skills + MCP: The Complete Picture

Skills and MCP serve complementary roles:

| | Skills | MCP |
|--|--------|-----|
| **What** | Structured instructions ("how to do X") | External tools ("ability to do X") |
| **Format** | Markdown files (SKILL.md) | Protocol + server process |
| **Loaded** | On demand (when invoked) | Always available (connected servers) |
| **Example** | "/ui-review" — checklist for reviewing designs | `calculate_contrast_ratio()` — actual computation |
| **Analogy** | A recipe book | A kitchen with tools |

In a real AI agent, they work together:
- A **skill** says "When reviewing a design, check all color contrasts"
- An **MCP tool** does the actual contrast calculation

```
┌──────────────────────────────────────────────┐
│  Skill: /ui-review                           │
│  "Check contrast for all text/background"     │
│       │                                       │
│       ▼                                       │
│  MCP Tool: calculate_contrast_ratio()         │
│  → Returns: 12.63:1 ✅ PASSES AA             │
└──────────────────────────────────────────────┘
```

### 5.7 The MCP Ecosystem

| MCP Server | What It Does |
|-----------|-------------|
| `server-filesystem` | Read/write local files |
| `server-github` | Interact with GitHub repos, PRs, issues |
| `server-postgres` | Query PostgreSQL databases |
| `server-brave-search` | Web search |
| Custom servers (like ours!) | Any tool you build |

> **Browse the ecosystem:** https://github.com/modelcontextprotocol/servers

### 5.8 Discussion

1. **Skills vs. MCP:** When would you use a skill vs. an MCP tool?
   - *Expected answer:* Skills for instructions and workflows. MCP tools for computations, data access, and external APIs. They're complementary, not competing.

2. **Auto-discovery:** Why is auto-discovery important?
   - *Expected answer:* You add a new tool to the server → every client automatically sees it. No code changes needed on the client side.

3. **Standardization value:** Why does a protocol create more value than individual tools?
   - *Expected answer:* Network effects — every new MCP server works with every MCP client. Build once, use everywhere.

---
## Wrap-Up & Key Takeaways

### What We Covered

| Section | Concept | One-Line Summary |
|---------|---------|------------------|
| 1 | **From Prompts to Skills** | Reusable, structured instruction sets that replace ad-hoc prompts |
| 2 | **Using Existing Skills** | Install community skills (like `colleague-skill`) and use them instantly |
| 3 | **Skill Design Patterns** | Arguments, dynamic context, and invocation control make skills flexible |
| 4 | **Skill Composition** | Isolated execution keeps context clean; skills compose into pipelines |
| 5 | **MCP** | A universal standard so any tool works with any LLM |

### The Complete Architecture

```
┌─────────────────────────────────────────────────────────────┐
│                    AI AGENT                                 │
│                                                             │
│  ┌──────────────┐  ┌──────────────┐  ┌──────────────────┐ │
│  │    Skills     │  │  Subagents   │  │  MCP Tools       │ │
│  │  (SKILL.md)   │  │  (isolated   │  │  (external       │ │
│  │              │  │   workers)    │  │   capabilities)  │ │
│  │  How to do X │  │  Run X in    │  │  Actually do X   │ │
│  │  (playbook)  │  │  isolation   │  │  (compute, fetch)│ │
│  └──────────────┘  └──────────────┘  └──────────────────┘ │
│                                                             │
│  Packaged as:  Plugins (shareable bundles)                 │
│  Standard:     Agent Skills (skills) + MCP (tools)         │
└─────────────────────────────────────────────────────────────┘
```

### Design Principles

1. **Use what exists first** — Before writing a skill, check if someone has already built one
2. **Skills replace copy-paste** — If you type the same instructions twice, make it a skill
3. **Good descriptions drive discovery** — The AI uses skill descriptions to decide when to load them
4. **Isolate with subagents** — Don't pollute main context with exploration/research
5. **MCP standardizes tools** — Build once, use with any LLM
6. **Skills + MCP = complete agents** — Skills say *what to do*; MCP tools *actually do it*

### Bonus Challenge (Take-Home)

**Build a Design Review Plugin** with these skills:

1. `/accessibility-audit` — Full WCAG 2.1 AA audit using the contrast MCP tool
2. `/brand-consistency` — Check if a design matches brand guidelines (reference skill)
3. `/responsive-check` — Review responsive behavior across breakpoints
4. `/design-system-gen` — Generate a complete design system from a brief (task skill with args)

Package them as a plugin:
```
design-review-plugin/
├── .claude-plugin/
│   └── plugin.json
├── skills/
│   ├── accessibility-audit/SKILL.md
│   ├── brand-consistency/SKILL.md
│   ├── responsive-check/SKILL.md
│   └── design-system-gen/SKILL.md
└── .mcp.json    (connects to design-tools MCP server)
```

**Hint:** Combine Lab 09 patterns (multi-agent review) with Lab 10 patterns (skills + MCP tools) for the most powerful result.

In [14]:
# YOUR BONUS CODE HERE
# Build the Design Review Plugin
# ...